# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

In [6]:
import csv

### Parse the csvs

In [7]:
def parse_csv(line):
    return next(csv.reader([line]))

# Remove the header and convert each citation into a kv pair -> (CITING, CITED)
citations_rdd = (
    rddCitations
    .filter(lambda line: not line.startswith('"CITING"'))
    .map(parse_csv)
    .map(lambda row: (int(row[0]), int(row[1])))
)

# Parse and cache the original patent for the state lookup and final result
patent_rows = (
    rddPatents
    .filter(lambda line: not line.startswith('"PATENT"'))
    .map(parse_csv)
    .cache()
)

# Create a small lookup RDD:
# PATENT -> STATE
patent_states = patent_rows.map(
    lambda row: (
        int(row[0]),
        row[5] if row[5] != "" else None
    )
)

### Left join: find the state of the CITING patent

In [8]:
# citations_rdd is already keyed by CITING -> (CITING, CITED)
# patent_states is -> (PATENT, STATE)

# Goal: CITING, (CITED, CITING_STATE))
citing_join = citations_rdd.leftOuterJoin(patent_states)

### Key the data by the CITED patent

In [9]:
# Rekey by CITED so the next RDD join can look up the cited patent's state
# Before: (CITING, (CITED, CITING_STATE))
# After:  (CITED, (CITING, CITING_STATE))

def rekey_by_cited(record):
    # Current record format -> (CITING, (CITED, CITING_STATE))
    citing = record[0]
    cited = record[1][0]
    citing_state = record[1][1]
    # Goal -> (CITED, (CITING, CITING_STATE))
    return (cited, (citing, citing_state))

# Apply the function to every record in citing_join
by_cited = citing_join.map(rekey_by_cited)

In [10]:
# Join using CITED as the key to look up its state
# Result shape -> (CITED, ((CITING, CITING_STATE), CITED_STATE))

both_states = by_cited.leftOuterJoin(patent_states)

### Flatten result to remove tuples

In [11]:
# Convert nested join output into -> (CITING, CITED, CITING_STATE, CITED_STATE)

def flatten_states(record):
    cited = record[0]
    citing = record[1][0][0]
    citing_state = record[1][0][1]
    cited_state = record[1][1]
    return (citing, cited, citing_state, cited_state)

# Flatten the nested join result and cache it
joined = both_states.map(flatten_states).cache()

# take() is an action not transformation so this computes and stores the cached result
joined.take(10)

[(5808416, 3689766, 'MA', None),
 (4980562, 3689766, 'MA', None),
 (4804852, 3689766, 'MA', None),
 (4514636, 3689766, 'TX', None),
 (4000426, 3689766, None, None),
 (4033904, 3689766, 'CA', None),
 (4922106, 3689766, 'MA', None),
 (5309064, 3689766, 'MA', None),
 (4105924, 3689766, None, None),
 (4013262, 3689766, 'MA', None)]

### Filter by matching states and get count

In [12]:
# Keep only citations where both states exist and match.
def is_same_state(record):
    # record = (CITING, CITED, CITING_STATE, CITED_STATE)
    citing_state = record[2]
    cited_state = record[3]

    # Only count the citation if both states exist and match
    return (
        citing_state is not None and
        cited_state is not None and
        citing_state == cited_state
    )

# Convert every matching citation into (CITING, 1) so reduceByKey can add the matches for each patent.
def make_count_pair(record):
    # (100, 200, "CO", "CO") -> (100, 1)
    citing = record[0]
    return (citing, 1)


same_state_counts = (
    joined
    .filter(is_same_state)
    .map(make_count_pair)
    .reduceByKey(lambda count1, count2: count1 + count2)
    .cache()
)

# (PATENT, COUNT)
same_state_counts.take(10)

[(4438146, 1),
 (4391643, 2),
 (5750988, 12),
 (3998211, 1),
 (4230675, 1),
 (4687449, 2),
 (5856234, 37),
 (5578836, 22),
 (5502000, 5),
 (5482884, 23)]

### Join counts back to orginal patents

In [13]:
def key_patent_by_id(row):
    # Make PATENT the key so its count can be joined back.
    # Convert: original row -> (PATENT, original row)
    patent_id = int(row[0])
    return (patent_id, row)

def append_same_state_count(record):
    # After the left join -> (PATENT, (original_row, count))
    patent_row = record[1][0]
    count = record[1][1]

    # A missing count means this patent had no same state citations.
    if count is None:
        count = 0

    # Append the new count to the original patent row.
    return patent_row + [count]

result = (
    patent_rows
    .map(key_patent_by_id)
    .leftOuterJoin(same_state_counts)
    .map(append_same_state_count)
)

In [15]:
# row[-1] = last value is same state count; row[0] = original patent ID.
# Highest count first and smaller patent ID breaks ties.
top10 = result.takeOrdered(
    10,
    key=lambda row: (-row[-1], int(row[0]))
)

for row in top10:
    print(row)

['5959466', '1999', '14515', '1997', 'US', 'CA', '5310', '2', '', '326', '4', '46', '159', '0', '1', '', '0.6186', '', '4.8868', '0.0455', '0.044', '', '', 125]
['5983822', '1999', '14564', '1998', 'US', 'TX', '569900', '2', '', '114', '5', '55', '200', '0', '0.995', '', '0.7201', '', '12.45', '0', '0', '', '', 103]
['6008204', '1999', '14606', '1998', 'US', 'CA', '749584', '2', '', '514', '3', '31', '121', '0', '1', '', '0.7415', '', '5', '0.0085', '0.0083', '', '', 100]
['5952345', '1999', '14501', '1997', 'US', 'CA', '749584', '2', '', '514', '3', '31', '118', '0', '1', '', '0.7442', '', '5.1102', '0', '0', '', '', 98]
['5958954', '1999', '14515', '1997', 'US', 'CA', '749584', '2', '', '514', '3', '31', '116', '0', '1', '', '0.7397', '', '5.181', '0', '0', '', '', 96]
['5998655', '1999', '14585', '1998', 'US', 'CA', '', '1', '', '560', '1', '14', '114', '0', '1', '', '0.7387', '', '5.1667', '', '', '', '', 96]
['5936426', '1999', '14466', '1997', 'US', 'CA', '5310', '2', '', '326', 